# 06 — Bootstrap stability analysis

500 nonparametric bootstrap resamples of the primary network at the fixed EBIC-selected alpha, re-estimating the full network + centrality pipeline on each resample to assess reproducibility of node rankings and edge selection (Methods 2.11). Uses the same seed=42 as the rest of the pipeline.

## Environment setup

In [ ]:
import os, sys, subprocess
from pathlib import Path


def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "scripts").exists() and (PROJECT_ROOT / "outputs").exists()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if _in_colab():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "scikit-learn", "networkx", "statsmodels", "openpyxl"], check=True)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
from scripts._pipeline_common import *  # noqa: F401,F403

print("Project root:", PROJECT_ROOT)


In [ ]:
import json
import numpy as np, pandas as pd
from sklearn.covariance import GraphicalLasso

fd = pd.read_csv(PROJECT_ROOT / 'outputs' / '01_ingest_harmonize' / 'feature_dictionary_v2.csv')
primary = fd.loc[fd.included_primary, 'feature'].tolist()
cols = primary

Z = pd.read_csv(PROJECT_ROOT / 'outputs' / '02_primary_network_dataset' / 'primary_direct_features_z.csv')
primary_summary = json.loads((PROJECT_ROOT / 'outputs' / '04_primary_graphical_lasso' / 'primary_graphical_lasso_model_summary.json').read_text())
edges = pd.read_csv(PROJECT_ROOT / 'outputs' / '04_primary_graphical_lasso' / 'primary_graphical_lasso_partial_edges.csv')
selected_alpha = primary_summary['selected_alpha']

B = 500
rng = np.random.default_rng(SEED)
edge_counts = {(min(a, b), max(a, b)): 0 for a in cols for b in cols if a < b}
cent = []
for b in range(B):
    idx = rng.integers(0, len(Z), len(Z))
    Zb = Z.iloc[idx].reset_index(drop=True)
    try:
        m = GraphicalLasso(alpha=selected_alpha, max_iter=50, tol=1e-2).fit(Zb)
        pr = m.precision_
        d = np.sqrt(np.diag(pr))
        Pb = -pr / np.outer(d, d)
        np.fill_diagonal(Pb, 1)
        eb = edge_df_from_partial(Pb, cols)
        strength = {c: 0.0 for c in cols}
        bridge = {c: 0.0 for c in cols}
        for _, r in eb.iterrows():
            a, bnode = r.source, r.target
            key = (min(a, bnode), max(a, bnode))
            edge_counts[key] += 1
            strength[a] += r.abs_weight
            strength[bnode] += r.abs_weight
            if DOM[a] != DOM[bnode]:
                bridge[a] += r.abs_weight
                bridge[bnode] += r.abs_weight
        cent.append(pd.DataFrame({'bootstrap': b, 'feature': cols,
                                   'strength': [strength[c] for c in cols],
                                   'bridge_strength': [bridge[c] for c in cols]}))
    except Exception:
        continue

print(f'Completed bootstrap resamples: {len(cent)} / {B}')


### Aggregate and save

In [ ]:
out_dir = PROJECT_ROOT / 'outputs' / '06_bootstrap_stability'
out_dir.mkdir(parents=True, exist_ok=True)

ef = pd.DataFrame([
    {'source': k[0], 'target': k[1], 'selection_frequency': v / B,
     'primary_edge': int(((edges.source == k[0]) & (edges.target == k[1]) | ((edges.source == k[1]) & (edges.target == k[0]))).any())}
    for k, v in edge_counts.items()
]).sort_values('selection_frequency', ascending=False)
ef.to_csv(out_dir / 'edge_bootstrap_frequency_primary.csv', index=False)

CB = pd.concat(cent, ignore_index=True)
boot = CB.groupby('feature').agg(
    strength_median=('strength', 'median'),
    strength_p025=('strength', lambda x: np.quantile(x, .025)),
    strength_p975=('strength', lambda x: np.quantile(x, .975)),
    bridge_strength_median=('bridge_strength', 'median'),
    bridge_p025=('bridge_strength', lambda x: np.quantile(x, .025)),
    bridge_p975=('bridge_strength', lambda x: np.quantile(x, .975)),
).reset_index()
boot['domain'] = boot.feature.map(DOM)
boot_sorted = boot.sort_values('bridge_strength_median', ascending=False)
boot_sorted.to_csv(out_dir / 'centrality_bootstrap_summary_primary.csv', index=False)

top_bridge_bootstrap_median = boot_sorted.head(8)[[
    'feature', 'strength_median', 'strength_p025', 'strength_p975',
    'bridge_strength_median', 'bridge_p025', 'bridge_p975', 'domain',
]].to_dict('records')

(out_dir / 'bootstrap_stability_summary.json').write_text(json.dumps({
    'n_bootstrap_requested': B,
    'n_bootstrap_successful': len(cent),
    'n_failures': B - len(cent),
    'alpha_fixed_from_primary_selection': float(selected_alpha),
    'stable_edges_freq_ge_0_5': int((ef.selection_frequency >= .5).sum()),
    'stable_edges_freq_ge_0_75': int((ef.selection_frequency >= .75).sum()),
    'top_bridge_bootstrap_median': top_bridge_bootstrap_median,
}, indent=2), encoding='utf-8')

boot.sort_values('bridge_strength_median', ascending=False).head(8)
